# Data cleaning

Groupwise log-MAD screening of MIDR and MFA. Run all cells to export the cleaned workbook and display the cleaning summary.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

INPUT_PATH = Path("data/Data.xlsx")
OUTPUT_PATH = Path("data/Data_clean.xlsx")
REPORT_DIR = Path("results/Data_Cleaning")
ROBUST_Z_THRESHOLD = 3.5
PGA_TOLERANCE = 0.002
PGA_LEVELS = np.array([0.1, 0.2, 0.3, 0.4, 0.5])
CASE_COLUMNS = [
    "stories",
    "brace_arrangement",
    "first_stage_yield_ratio",
    "apdybrb_fy2_mean",
]
TARGET_COLUMNS = ["MIDR", "MFA"]


In [2]:
def calculate_robust_z(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    scores = pd.Series(np.nan, index=series.index, dtype=float)
    valid = np.isfinite(values) & values.gt(0)
    if not valid.any():
        return scores
    log_values = np.log(values.loc[valid])
    median = log_values.median()
    mad = np.median(np.abs(log_values - median))
    if not np.isfinite(mad) or mad <= np.finfo(float).eps:
        scores.loc[valid] = 0.0
    else:
        scores.loc[valid] = 0.67448975 * (log_values - median) / mad
    return scores


def clean_dataset(
    input_path: Path,
    output_path: Path,
    threshold: float = ROBUST_Z_THRESHOLD,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not np.isfinite(threshold) or threshold <= 0:
        raise ValueError("The robust-z threshold must be finite and positive.")
    if input_path.resolve() == output_path.resolve():
        raise ValueError("Input and output paths must differ.")
    data = pd.read_excel(input_path, sheet_name="Sheet1")
    original_columns = data.columns.tolist()
    required = CASE_COLUMNS + ["PGA (g)"] + TARGET_COLUMNS
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")
    data["Source_excel_row"] = np.arange(2, len(data) + 2)

    pga = pd.to_numeric(data["PGA (g)"], errors="coerce").to_numpy(dtype=float)
    finite_pga = np.isfinite(pga)
    levels = np.full(len(data), np.nan)
    errors = np.full(len(data), np.nan)
    distances = np.abs(pga[finite_pga, None] - PGA_LEVELS[None, :])
    levels[finite_pga] = PGA_LEVELS[distances.argmin(axis=1)]
    errors[finite_pga] = distances.min(axis=1)
    invalid_pga = ~finite_pga | (errors > PGA_TOLERANCE)
    levels[invalid_pga] = np.nan
    data["PGA_level"] = levels
    data["PGA_level_error"] = errors
    data["Invalid_case"] = data[CASE_COLUMNS].isna().any(axis=1)
    data["Invalid_PGA"] = invalid_pga
    data["Invalid_EDP"] = False

    for target in TARGET_COLUMNS:
        values = pd.to_numeric(data[target], errors="coerce")
        data["Invalid_EDP"] |= ~np.isfinite(values) | values.le(0)
        data[f"Robust_z_{target}"] = data.groupby(
            CASE_COLUMNS + ["PGA_level"], dropna=False
        )[target].transform(calculate_robust_z)
        data[f"{target}_outlier"] = data[f"Robust_z_{target}"].abs().gt(threshold)

    flags = [
        "Invalid_case",
        "Invalid_PGA",
        "Invalid_EDP",
        "MIDR_outlier",
        "MFA_outlier",
    ]
    removed = data[flags].any(axis=1)
    cleaned = data.loc[~removed, original_columns].copy()
    report_columns = [
        "Source_excel_row",
        *CASE_COLUMNS,
        "PGA (g)",
        "PGA_level",
        "PGA_level_error",
        *TARGET_COLUMNS,
        "Robust_z_MIDR",
        "Robust_z_MFA",
        *flags,
    ]
    report = data.loc[removed, report_columns].copy()
    summary = pd.DataFrame(
        {
            "Item": [
                "Robust z threshold",
                "Original samples",
                "Missing structural identifiers",
                "Invalid PGA values",
                "Invalid EDP values",
                "MIDR statistical outliers",
                "MFA statistical outliers",
                "Removed samples",
                "Removed percentage",
                "Retained samples",
            ],
            "Value": [
                threshold,
                len(data),
                *[int(data[flag].sum()) for flag in flags],
                int(removed.sum()),
                100 * removed.mean() if len(data) else 0.0,
                len(cleaned),
            ],
        }
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cleaned.to_excel(output_path, sheet_name="Sheet1", index=False)
    saved = pd.read_excel(output_path, sheet_name="Sheet1")
    pd.testing.assert_frame_equal(
        cleaned.reset_index(drop=True), saved, check_dtype=False, rtol=1e-12, atol=0
    )
    return cleaned, report, summary


In [3]:
data_clean, removed_samples, cleaning_summary = clean_dataset(
    INPUT_PATH, OUTPUT_PATH, ROBUST_Z_THRESHOLD
)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
cleaning_summary.to_csv(REPORT_DIR / "cleaning_summary.csv", index=False)
removed_samples.to_csv(REPORT_DIR / "removed_samples.csv", index=False)
settings = pd.DataFrame(
    {
        "Setting": [
            "Robust z threshold",
            "PGA tolerance (g)",
            "PGA levels (g)",
            "Group columns",
        ],
        "Value": [
            str(ROBUST_Z_THRESHOLD),
            str(PGA_TOLERANCE),
            ", ".join(map(str, PGA_LEVELS)),
            ", ".join(CASE_COLUMNS),
        ],
    }
)
settings.to_csv(REPORT_DIR / "cleaning_settings.csv", index=False)
display(cleaning_summary.round({"Value": 4}))
print(f"Cleaned data: {OUTPUT_PATH}")
print(f"Cleaning reports: {REPORT_DIR}")


,Item,Value
0,Robust z threshold,3.5000
1,Original samples,24000.0000
2,Missing structural identifiers,0.0000
3,Invalid PGA values,0.0000
4,Invalid EDP values,0.0000
5,MIDR statistical outliers,55.0000
6,MFA statistical outliers,75.0000
7,Removed samples,128.0000
8,Removed percentage,0.5333
9,Retained samples,23872.0000


Cleaned data: data\Data_clean.xlsx
Cleaning reports: results\Data_Cleaning


The modified-z rule and its interpretation follow [NIST/SEMATECH, Section 1.3.5.17](https://www.itl.nist.gov/div898/handbook/eda/section3/eda35h.htm), which cites Iglewicz and Hoaglin (1993), *How to Detect and Handle Outliers*. PGA tolerance and structural grouping are dataset settings. The accompanying cleaning documentation gives the formula, zero-MAD convention and exclusion policy.